In [0]:
%sql
SELECT 'bronze_bls_all_data' AS table_name, COUNT(*) AS record_count
FROM bls.bronze.bronze_bls_all_data

UNION ALL

SELECT 'bronze_bls_series', COUNT(*)
FROM bls.bronze.bronze_bls_series

UNION ALL

SELECT 'bronze_bls_sector', COUNT(*)
FROM bls.bronze.bronze_bls_sector

UNION ALL

SELECT 'bronze_bls_class', COUNT(*)
FROM bls.bronze.bronze_bls_class

UNION ALL

SELECT 'bronze_bls_measure', COUNT(*)
FROM bls.bronze.bronze_bls_measure

UNION ALL

SELECT 'bronze_bls_duration', COUNT(*)
FROM bls.bronze.bronze_bls_duration

UNION ALL

SELECT 'bronze_bls_seasonal', COUNT(*)
FROM bls.bronze.bronze_bls_seasonal

UNION ALL

SELECT 'bronze_population', COUNT(*)
FROM bls.bronze.bronze_population;

In [0]:

%sql
SELECT COUNT(*) AS silver_count
FROM bls.bronze.silver_bls_data;

SELECT
    series_id,
    year,
    period,
    COUNT(*) AS record_count
FROM bls.bronze.silver_bls_data
GROUP BY series_id, year, period
HAVING COUNT(*) > 1
ORDER BY record_count DESC;


In [0]:
%sql
SELECT
    SUM(CASE WHEN series_id IS NULL THEN 1 ELSE 0 END) AS null_series_id,
    SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END) AS null_year,
    SUM(CASE WHEN period IS NULL THEN 1 ELSE 0 END) AS null_period,
    SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS null_value,
    SUM(CASE WHEN sector_name IS NULL THEN 1 ELSE 0 END) AS null_sector_name,
    SUM(CASE WHEN class_text IS NULL THEN 1 ELSE 0 END) AS null_class_text,
    SUM(CASE WHEN measure_text IS NULL THEN 1 ELSE 0 END) AS null_measure_text
FROM bls.bronze.silver_bls_data;

In [0]:
%sql
SELECT COUNT(*) AS unmatched_series
FROM bls.bronze.silver_bls_data
WHERE sector_name IS NULL
   OR class_text IS NULL
   OR measure_text IS NULL;

In [0]:
%sql
SELECT
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    COUNT(DISTINCT year) AS distinct_years,
    COUNT(DISTINCT period) AS distinct_periods
FROM bls.bronze.silver_bls_data;

In [0]:
%sql
SELECT
    period,
    COUNT(*) AS record_count
FROM bls.bronze.silver_bls_data
GROUP BY period
ORDER BY period;

In [0]:
%sql
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT series_id) AS distinct_series
FROM bls.gold.gold_bls_best_year;

In [0]:
%sql
DROP TABLE IF EXISTS bls.bronze.silver_population;
DROP TABLE IF EXISTS bls.bronze.silver_bls_data;
DROP TABLE IF EXISTS bls.bronze.gold_population_statistics;

In [0]:
period_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", "\t")
    .load("/Volumes/bls/bronze/bls_raw/raw/pr.period")
)

display(period_df)

In [0]:
contacts_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", "\t")
    .load("/Volumes/bls/bronze/bls_raw/raw/pr.contacts")
)
display(contacts_df)

In [0]:
footnote_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", "\t")
    .load("/Volumes/bls/bronze/bls_raw/raw/pr.footnote")
)
display(footnote_df)

In [0]:
Current_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", "\t")
    .load("/Volumes/bls/bronze/bls_raw/raw/pr.data.0.Current")
)
display(Current_df)

In [0]:
txt_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", "\t")
    .load("/Volumes/bls/bronze/bls_raw/raw/pr.txt")
)
display(txt_df)

In [0]:
%sql
SELECT
    series_id,
    year,
    MAX(CASE WHEN period = 'Q05' THEN value END) AS q05,
    AVG(CASE WHEN period IN ('Q01','Q02','Q03','Q04') THEN value END) AS avg_q01_q04
FROM bls.silver.silver_bls_data
WHERE series_id = 'PRS30006032'
GROUP BY series_id, year
ORDER BY year;

In [0]:
%sql
SELECT *
FROM bls.gold.gold_bls_best_year
ORDER BY series_id
LIMIT 20;

In [0]:
%sql
SELECT COUNT(*) AS series_count
FROM bls.gold.gold_bls_best_year;

In [0]:
%sql
SELECT
    COUNT(DISTINCT series_id) AS total_series,
    COUNT(DISTINCT CASE
        WHEN period IN ('Q01', 'Q02', 'Q03', 'Q04')
        THEN series_id
    END) AS series_with_quarters,
    COUNT(DISTINCT CASE
        WHEN period = 'Q05'
        THEN series_id
    END) AS series_with_q05
FROM bls.silver.silver_bls_data;

In [0]:
%sql
SELECT
    series_id,
    series_label
FROM bls.silver.silver_bls_data
GROUP BY
    series_id,
    series_label
HAVING
    COUNT(CASE WHEN period IN ('Q01', 'Q02', 'Q03', 'Q04') THEN 1 END) = 0
ORDER BY series_id;

In [0]:
%sql
SELECT DISTINCT
    series_id,
    series_label
FROM bls.silver.silver_bls_data
WHERE series_id IN (
    'PRS30006081','PRS30006082','PRS30006083',
    'PRS30006131','PRS30006132','PRS30006133',
    'PRS30006171','PRS30006172','PRS30006173',
    'PRS30006221','PRS30006222','PRS30006223',
    'PRS30006231','PRS30006232','PRS30006233',
    'PRS31006081','PRS31006082','PRS31006083',
    'PRS31006131','PRS31006132','PRS31006133',
    'PRS31006171','PRS31006172','PRS31006173',
    'PRS31006221','PRS31006222','PRS31006223',
    'PRS31006231','PRS31006232','PRS31006233',
    'PRS32006081','PRS32006082','PRS32006083',
    'PRS32006131','PRS32006132','PRS32006133',
    'PRS32006171','PRS32006172','PRS32006173',
    'PRS32006221','PRS32006222','PRS32006223',
    'PRS32006231','PRS32006232','PRS32006233'
)
ORDER BY series_id;

In [0]:
%sql
SELECT
    series_id,
    COUNT(*) AS cnt
FROM bls.gold.gold_bls_best_year
GROUP BY series_id
HAVING COUNT(*) <> 1;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM bls.gold.gold_series_population;

In [0]:
%sql
SELECT *
FROM bls.gold.gold_series_population
ORDER BY year;

In [0]:
%sql
SELECT *
FROM bls.gold.gold_population_statistics;

In [0]:
%sql
SELECT
    CONCAT('[', series_id, ']') AS series_id_visible,
    LENGTH(series_id) AS original_length,
    LENGTH(TRIM(series_id)) AS trim_length,
    LENGTH(REGEXP_REPLACE(series_id, '\\s+', '')) AS whitespace_removed_length
FROM bls.bronze.bronze_bls_all_data
WHERE series_id LIKE '%PRS30006032%';

In [0]:
%sql
select * from bls.gold.gold_series_population